# 06 — Stage 1: Predictions & Country-Year Shadow Measure

Generate intervention probability predictions for all directed dyad-years and
aggregate to produce the country-year shadow measure.

**Inputs**:
- `data/interim/dd_spat_{cy}_{ud}.parquet` — feature data (from nb04)
- `data/interim/sl_model_{cy}_{ud}.pkl`   — fitted models (from nb05)
- `data/interim/sl_oof_{cy}_{ud}.parquet` — OOF predictions for onset rows

**Outputs**:
- `data/interim/cy_shadow_{cy}_{ud}.parquet` — country-year shadow variables:
  - `E_gov`, `E_opp` (raw expected-count sums)
  - `E_gov_trim`, `E_opp_trim` (trimmed at τ = 0.001)
  - `E_*_asinh` (asinh-transformed variants of the above)
  - Type-disaggregated: `E_major_*`, `E_P5_*`, `E_contig_*`,
    `E_coethnic_*`, `E_colonial_*`, `E_rival_*`, `E_hostile_*`, `E_doe_*`
- `data/interim/sl_preds_{cy}_{ud}.parquet` — per-dyad predictions
  (ccode_A, ccode_B, year, p_gov, p_opp)
- `data/interim/sl_fp_diag.parquet` — fixed-point convergence diagnostics
  (all 25 draws consolidated; keyed by cy, ud, fp_iter)

**Reference R scripts**: `15-generatePredictions.R`, `16-makePirate.R`

**Key design choices**:
- For Regan-period onset rows, the OOF predictions from notebook 05 are
  used (avoiding train-on-predict leakage).  For all other rows, the
  full-data model predicts directly.
- The ex post Nash fixed-point iterates predictions → spatial lags → predictions
  until convergence, ensuring self-consistent equilibrium across the full panel.
- Type-disaggregated aggregation slices the converged predictions by intervener
  characteristics (major power, contiguity, ethnicity, colonial ties, rivalry,
  DOE military capability).

**Cutpoint**: τ = 0.001 (tuned in Stage 2; see constructing.typ §Aggregating).

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve() / "src"))
from shadow.data.spatial import update_spatial_lags_proba, _build_W_cache

ROOT    = Path("..").resolve()
INTERIM = ROOT / "data" / "interim"

TAU         = 0.001   # cutpoint for trimming low-probability interveners
MAX_FP_ITER = 10      # fixed-point iterations (raised from 5 for universal scope)
FP_TOL      = 5e-4    # convergence: mean |Δ spat| (consistent with nb05 burnout)

# DGG peace scale threshold: ≤ 0.25 = severe + lesser rivalry
RIVAL_THRESHOLD = 0.25


In [2]:
# Reuse feature column logic from nb05
ID_COLS = [
    "ccode_A", "ccode_B", "year", "ddyear",
    "onset_A", "regan_period", "intervention",
]

def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = set(ID_COLS)
    return [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]


def predict_proba_from_model(model_result: dict, X: np.ndarray) -> np.ndarray:
    """Use the fitted super-learner ensemble to predict P(class) for X."""
    scaler  = model_result["scaler"]
    pca     = model_result["pca"]
    weights = model_result["weights"]
    clfs    = model_result["classifiers"]

    X_sc = scaler.transform(X)
    X_pc = pca.transform(X_sc)

    proba = np.zeros((len(X), 3))
    for name, clf in clfs.items():
        proba += weights[name] * clf.predict_proba(X_pc)
    return proba


## Main loop: Nash-equilibrium predictions for all (B, A, t)

For each imputation draw, we solve the universal fixed-point equation:

    σ*(B, A, t) = M(X(B, A, t, σ*); θ)  for all (B, A, t)

The trained model θ is held fixed.  We iterate:
  predict → update spatial lags (all rows) → re-predict → … → converge.

This gives Nash-consistent predictions even for non-onset years: the
shadow measure E_gov(A, t) for a pre-conflict year reflects the counterfactual
equilibrium that would obtain if a conflict started, with all other potential
interveners playing their model-equilibrium strategies.

**Convergence**: tracked via `mean |Δ spat_gov, spat_opp|` (consistent with
the nb05 burnout metric) and `mean |Δ proba|`.  Tolerance = 5e-4 on delta_spat,
up to 10 iterations.  Per-iteration diagnostics saved to `sl_fp_diag.parquet`.

**Aggregation**: after convergence + OOF splice, predictions are aggregated
to country-year both unconditionally and by intervener type (major power,
P5, contiguous, co-ethnic, colonial ruler, rivalry, hostility-weighted).


In [ ]:
fp_diag_rows = []  # collect convergence diagnostics across all draws

for cy in range(1, 6):
    for ud in range(1, 6):
        print(f"── CY {cy}/5, UD {ud}/5 ", end="", flush=True)

        dd    = pd.read_parquet(INTERIM / f"dd_spat_{cy}_{ud}.parquet")
        model = joblib.load(INTERIM / f"sl_model_{cy}_{ud}.pkl")
        oof   = pd.read_parquet(INTERIM / f"sl_oof_{cy}_{ud}.parquet")

        feat_cols = get_feature_cols(dd)
        onset_mask = dd["onset_A"] == 1

        # Build W cache for ALL years (not just onset years): the universal
        # fixed-point updates spatial lags for every (A, t), onset or not.
        all_mask = pd.Series(True, index=dd.index)
        W_cache  = _build_W_cache(dd, all_mask)

        # ── Universal Nash fixed-point ─────────────────────────────────────
        # Solve σ*(B,A,t) = M(X(B,A,t,σ*); θ) for ALL (B,A,t).
        # The trained model θ is fixed; only the spatial lags change.
        prev_proba = None
        prev_spat  = None
        converged  = False

        for fp_iter in range(MAX_FP_ITER):
            X_all     = dd[feat_cols].fillna(0).to_numpy(dtype=float)
            proba_all = predict_proba_from_model(model, X_all)

            # Track both convergence metrics.
            delta_proba = 0.0
            delta_spat_onset = 0.0
            delta_spat_all   = 0.0

            if prev_proba is not None:
                delta_proba = float(np.abs(proba_all - prev_proba).mean())

            cur_spat = dd[["spat_gov", "spat_opp"]].fillna(0).to_numpy()
            if prev_spat is not None:
                diff_spat = np.abs(cur_spat - prev_spat)
                delta_spat_all = float(diff_spat.mean())
                if onset_mask.any():
                    delta_spat_onset = float(diff_spat[onset_mask.values].mean())

            fp_diag_rows.append({
                "cy": cy, "ud": ud, "fp_iter": fp_iter,
                "delta_proba": delta_proba,
                "delta_spat_onset": delta_spat_onset,
                "delta_spat_all": delta_spat_all,
                "converged": False,
            })

            # Check convergence on delta_spat_all (consistent with nb05 burnout).
            if prev_spat is not None and delta_spat_all < FP_TOL:
                fp_diag_rows[-1]["converged"] = True
                converged = True
                break

            prev_proba = proba_all.copy()
            prev_spat  = cur_spat.copy()

            # Update spatial lags for ALL rows (onset_only=False).
            dd = update_spatial_lags_proba(
                dd,
                proba_all[:, 1],   # p_gov
                proba_all[:, 2],   # p_opp
                W_cache=W_cache,
                onset_only=False,
            )

        tag = "converged" if converged else "max-iter"
        print(f"(FP: {fp_iter + 1} iters, {tag}, "
              f"Δspat={delta_spat_all:.2e}) ", end="", flush=True)

        # ── Assign predictions ─────────────────────────────────────────────
        dd["p_none"] = proba_all[:, 0]
        dd["p_gov"]  = proba_all[:, 1]
        dd["p_opp"]  = proba_all[:, 2]

        # Overwrite Regan-period onset rows with OOF predictions (no leakage).
        oof_keys = set(oof["ddyear"].values)
        oof_mask = dd["ddyear"].isin(oof_keys)
        if oof_mask.any():
            oof_lookup = oof.set_index("ddyear")[["p_none", "p_gov", "p_opp"]]
            for col in ["p_none", "p_gov", "p_opp"]:
                dd.loc[oof_mask, col] = (
                    dd.loc[oof_mask, "ddyear"].map(oof_lookup[col]).values
                )

        # ── Save per-dyad predictions (lightweight) ────────────────────────
        dd[["ccode_A", "ccode_B", "year", "p_gov", "p_opp"]].to_parquet(
            INTERIM / f"sl_preds_{cy}_{ud}.parquet", index=False
        )

        # ── Aggregate to country-year ──────────────────────────────────────
        def agg(group):
            pg = group["p_gov"].values
            po = group["p_opp"].values
            result = {
                "E_gov":      pg.sum(),
                "E_opp":      po.sum(),
                "E_gov_trim": pg[pg >= TAU].sum(),
                "E_opp_trim": po[po >= TAU].sum(),
                "n_B":        len(group),
            }

            # ── Type-disaggregated aggregations ────────────────────────────
            type_filters = {
                "major":    group["major_power_B"].values == 1,
                "P5":       group["is_P5_B"].values == 1,
                "contig":   group["ud_conttype"].values <= 5,
                "coethnic": group["ud_sameFirstEth"].values == 1,
                "colonial": group["B_wasColOf_A"].values == 1,
                "rival":    group["ud_peace"].values <= RIVAL_THRESHOLD,
            }

            for prefix, mask in type_filters.items():
                pg_m = pg[mask]
                po_m = po[mask]
                result[f"E_{prefix}_gov"]      = pg_m.sum()
                result[f"E_{prefix}_opp"]      = po_m.sum()
                result[f"E_{prefix}_gov_trim"] = pg_m[pg_m >= TAU].sum()
                result[f"E_{prefix}_opp_trim"] = po_m[po_m >= TAU].sum()

            # Hostility-weighted (continuous): weight = (1 - ud_peace)
            hostility = 1.0 - group["ud_peace"].values
            result["E_hostile_gov"]      = (pg * hostility).sum()
            result["E_hostile_opp"]      = (po * hostility).sum()
            pg_h = pg * hostility
            po_h = po * hostility
            result["E_hostile_gov_trim"] = pg_h[pg >= TAU].sum()
            result["E_hostile_opp_trim"] = po_h[po >= TAU].sum()

            # DOE-weighted (continuous): weight = doe_pr_win_B
            # (Carroll/Kenkel dispute outcome expectations — B's probability
            # of defeating A militarily)
            doe_w = group["doe_pr_win_B"].values
            result["E_doe_gov"]      = (pg * doe_w).sum()
            result["E_doe_opp"]      = (po * doe_w).sum()
            pg_d = pg * doe_w
            po_d = po * doe_w
            result["E_doe_gov_trim"] = pg_d[pg >= TAU].sum()
            result["E_doe_opp_trim"] = po_d[po >= TAU].sum()

            return pd.Series(result)

        cy_shadow = (
            dd.groupby(["ccode_A", "year"]).apply(agg, include_groups=False)
            .reset_index()
            .rename(columns={"ccode_A": "ccode"})
        )

        # asinh transforms for all E_* columns (raw + trimmed).
        e_cols = [c for c in cy_shadow.columns if c.startswith("E_")]
        for col in e_cols:
            cy_shadow[f"{col}_asinh"] = np.arcsinh(cy_shadow[col])

        cy_shadow["cy_imp"] = cy
        cy_shadow["ud_imp"] = ud
        cy_shadow.to_parquet(INTERIM / f"cy_shadow_{cy}_{ud}.parquet", index=False)

        print(f"-> {len(cy_shadow):,} cy rows, "
              f"{len(e_cols)} E_* cols, "
              f"E_gov_asinh mean={cy_shadow['E_gov_asinh'].mean():.3f}")

# ── Save convergence diagnostics ───────────────────────────────────────────
fp_diag = pd.DataFrame(fp_diag_rows)
fp_diag.to_parquet(INTERIM / "sl_fp_diag.parquet", index=False)
print(f"\nFP diagnostics: {len(fp_diag)} rows saved")
print(fp_diag.groupby(["cy", "ud"])["fp_iter"].max().describe())

# ── Final assertions ───────────────────────────────────────────────────────
n_shadow = len(list(INTERIM.glob("cy_shadow_*.parquet")))
n_preds  = len(list(INTERIM.glob("sl_preds_*.parquet")))
print(f"\nShadow files: {n_shadow}  (expected 25)")
print(f"Preds files:  {n_preds}  (expected 25)")
assert n_shadow == 25, f"Expected 25 shadow files, got {n_shadow}"
assert n_preds == 25, f"Expected 25 preds files, got {n_preds}"